# August 2026 verification · **09 — The chart and the table**

The two visuals the article carries, drawn from Gold and nothing else:

1. **The chart.** Total list, over 18 weeks and over 52 weeks on one
   indexed axis, **March 2026 = 100**, April 2025 to August 2026, with the
   target deadline marked. Indexing puts three series of very different size
   on one honest scale, with no second axis. The full window stays in, so the
   reader sees both the fall into the deadline and the climb after it. A
   chart starting in March would hide the first half of that.
2. **The three-row table.** List / over 18 / over 52 for August 2026: since
   March, against July, against August 2025, absolute and %. The columns sit
   side by side so the reader sees the denominator effect for themselves.

**Basis:** NHS England's headline (including its estimates for trusts that
didn't submit), the basis locked for national figures in the article. The
chart's footnote states that the finding holds on my reported measure and on
a constant panel too, with the numbers read from Gold.

**Style:** the same ink, grid, surface and categorical slots as the Two Maps
chart, so the series reads as one body of work. Slots 1-3 (blue, orange,
aqua) are the reference palette's all-pairs-validated set. Aqua is below 3:1
contrast on the light surface, so every line is directly labelled and the
table and CSV are the text view. Medium renders on white, so these are
light-mode only.

Outputs: `data/gold/09_charts/`.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR  # noqa: E402

OUT = GOLD_DIR / "09_charts"
OUT.mkdir(parents=True, exist_ok=True)

# Two Maps' tokens, reused deliberately.
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#9a9891", "#e6e4de"
SERIES = {"over_52": "#2a78d6", "total": "#eb6834", "over_18": "#1baf7a"}   # slots 1, 2, 3
LABEL = {"over_52": "Waiting over 52 weeks", "total": "Total waiting list", "over_18": "Waiting over 18 weeks"}
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titlesize": 14, "axes.unicode_minus": True})

def signed(x: float, fmt: str) -> str:
    """Format with an explicit sign, using a true minus rather than a hyphen."""
    return format(x, "+" + fmt).replace("-", "\u2212")

## Step 0 — Upstream must be present, and the gate must have passed

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), "reconciliation gate failed: run 04 first"
idx = pd.read_csv(GOLD_DIR / "05_divergence" / "index_long.csv")
dv = pd.read_csv(GOLD_DIR / "05_divergence" / "divergence_summary.csv").set_index(["window", "basis"])
t3 = pd.read_csv(GOLD_DIR / "05_divergence" / "three_row_table.csv")
tg = pd.read_csv(GOLD_DIR / "05_divergence" / "march_targets.csv", index_col=0)

h = idx[(idx.window == "Mar26") & (idx.basis == "headline")]
# index_long only holds March onwards for the Mar26 window; the chart needs the
# full run, so I re-base the April-2025-window headline series to March here.
full = idx[(idx.window == "Apr25") & (idx.basis == "headline")].set_index("period")[["total", "over_18", "over_52"]]
chart = 100 * full / full.loc["2026-03"]
# Must agree with notebook 05's own March-based index wherever both exist.
chk = (chart.loc[h.period] - h.set_index("period")[["total_index", "over_18_index", "over_52_index"]].set_axis(chart.columns, axis=1)).abs()
assert (chk < 1e-9).all().all()
chart.index = pd.PeriodIndex(chart.index, freq="M").to_timestamp()
chart.round(1).tail(6)

,total,over_18,over_52
period,,,
2026-03-01,100.0,100.0,100.0
2026-04-01,101.6,102.6,105.7
2026-05-01,102.4,101.7,110.9
2026-06-01,102.3,100.8,112.0
2026-07-01,103.1,102.8,118.0
2026-08-01,103.7,104.0,120.1


## Step 1 — The chart

In [3]:
fig, ax = plt.subplots(figsize=(10, 6.4), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
mar = pd.Timestamp("2026-03-01")

ax.axhline(100, color=MUTED, lw=1, ls=(0, (2, 2)), zorder=1)
ax.axvline(mar, color=INK2, lw=1, zorder=1)
t = tg.loc["2026-03"]
ax.text(mar, 214, "  March 2026: target deadline\n  65% within 18 weeks (met: "
        f"{t.pct_within_18:.1f}%)\n  under 1% over 52 weeks (missed: {t.pct_over_52:.2f}%)",
        fontsize=9, color=INK2, va="top", ha="left")

for key in ["total", "over_18", "over_52"]:
    lw = 2.6 if key == "over_52" else 2
    ax.plot(chart.index, chart[key], color=SERIES[key], lw=lw, solid_capstyle="round", solid_joinstyle="round", zorder=3)
    ax.scatter(chart.index[-1], chart[key].iloc[-1], s=36, color=SERIES[key], edgecolor=SURFACE, lw=1.5, zorder=4)

# Direct labels at the line ends. Total and over-18 end almost on top of each
# other (both about 104), so they get short leader lines to labels set apart,
# rather than nudged labels floating free of their lines.
end = chart.index[-1]
lab_x = end + pd.Timedelta(days=22)
since = {k: chart[k].iloc[-1] - 100 for k in chart}
placements = {"over_52": 123, "over_18": 109, "total": 95}
for key, y_lab in placements.items():
    y_end = chart[key].iloc[-1]
    if abs(y_lab - y_end) > 0.5:
        ax.plot([end + pd.Timedelta(days=6), lab_x - pd.Timedelta(days=4)], [y_end, y_lab], color=MUTED, lw=0.8, zorder=2)
    ax.text(lab_x, y_lab, f"{LABEL[key]}\n{signed(since[key], '.1f')}% since March", fontsize=9.5, color=INK, va="center", ha="left")

# One selective annotation: the month the tail fell into the deadline.
feb, m26 = chart.over_52.loc["2026-02-01"], chart.over_52.loc["2026-03-01"]
seg_x = pd.Timestamp("2026-02-15"); seg_y = (feb + m26) / 2
# The note sits in the empty band between the over-18 line and the falling
# over-52 line, so its short leader crosses nothing.
ax.annotate(f"Over 52 weeks:\n{signed(100 * (m26 / feb - 1), '.0f')}% in one month", xy=(seg_x, seg_y),
            xytext=(pd.Timestamp("2025-11-01"), 127), va="center",
            fontsize=9, color=INK2, arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))

ax.set_xlim(chart.index[0] - pd.Timedelta(days=12), end + pd.Timedelta(days=150))
ax.set_ylim(75, 218)
ax.set_xticks(pd.date_range("2025-04-01", "2026-08-01", freq="2MS"))
ax.set_xticklabels([d.strftime("%b\n%Y") if d.month in (4,) or d == chart.index[0] else d.strftime("%b") for d in
                    pd.date_range("2025-04-01", "2026-08-01", freq="2MS")], fontsize=9, color=INK2)
ax.tick_params(axis="y", labelsize=9, colors=INK2, length=0)
ax.tick_params(axis="x", length=0)
ax.set_ylabel("Index, March 2026 = 100", fontsize=9.5, color=INK2)
ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(GRID)

hd, pn, rp = dv.loc[("Mar26", "headline")], dv.loc[("Mar26", "panel")], dv.loc[("Mar26", "reported")]
fig.text(0.02, 0.965, "Since March's target deadline, waits of over a year have grown every month",
         fontsize=14.5, color=INK, ha="left", va="top", weight="bold")
fig.text(0.02, 0.918, f"Incomplete RTT pathways, England, April 2025 to August 2026. Since March: over 52 weeks "
         f"{signed(hd.over_52_change_pct, '.1f')}%, total list {signed(hd.total_change_pct, '.1f')}%.",
         fontsize=10, color=INK2, ha="left", va="top")
fig.text(0.02, 0.062, f"Headline basis: NHS England figures including its estimates for trusts that did not submit. The pattern holds on "
         f"reported providers only ({signed(rp.over_52_change_pct, '.1f')}% vs {signed(rp.total_change_pct, '.1f')}%)\nand on a constant panel of trusts "
         f"that reported every month ({signed(pn.over_52_change_pct, '.1f')}% vs {signed(pn.total_change_pct, '.1f')}%).",
         fontsize=7.8, color=INK2, ha="left", va="bottom")
fig.text(0.02, 0.012, "Source: NHS England, RTT full CSV extracts Apr 2025 \u2013 Aug 2026 and missing-data adjustments (Aug 2026 release).\n"
         "Rebuilt from provider files and reconciled exactly to NHS England's published headline.",
         fontsize=7.8, color=INK2, ha="left", va="bottom")
fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.19)
fig.savefig(OUT / "divergence_since_march.png", dpi=200, facecolor=SURFACE)
plt.close(fig)
print("wrote", (OUT / "divergence_since_march.png").name)

wrote divergence_since_march.png


## Step 2 — The three-row table

Drawn as text on a grid rather than with matplotlib's table widget, so the
figures can be right-aligned on their digits. Text is ink only; colour stays
on the chart's lines.

In [4]:
hb = t3[t3.basis == "headline"].set_index("measure").loc[["Total list", "Over 18 weeks", "Over 52 weeks"]]
cols = [("August 2026", None), ("Since March 2026", "since_mar"), ("vs July 2026", "mom"), ("vs August 2025", "yoy")]

def cell(r, key):
    return f"{signed(r[key + '_abs'], ',.0f')}  ({signed(r[key + '_pct'], '.1f')}%)"

fig, ax = plt.subplots(figsize=(10, 2.6), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE); ax.axis("off")
xs = [0.02, 0.43, 0.63, 0.81, 0.995]          # left edge of label column, then right edges of number columns
ys = [0.80, 0.56, 0.36, 0.16]
ax.text(xs[0], ys[0], "England, incomplete pathways", fontsize=10, color=INK2, va="center", transform=ax.transAxes)
for j, (head, _) in enumerate(cols):
    ax.text(xs[j + 1] - (0.0 if j else 0.0), ys[0], head, fontsize=10, color=INK2, va="center", ha="right", weight="bold",
            transform=ax.transAxes)
ax.plot([0.02, 0.995], [0.70, 0.70], color=INK2, lw=0.8, transform=ax.transAxes)
for i, (name, r) in enumerate(hb.iterrows()):
    y = ys[i + 1]
    ax.text(xs[0], y, name, fontsize=11, color=INK, va="center", transform=ax.transAxes,
            weight="bold" if name == "Over 52 weeks" else "normal")
    ax.text(xs[1], y, f"{r.aug_2026:,.0f}", fontsize=11, color=INK, va="center", ha="right", transform=ax.transAxes)
    for j, (_, key) in enumerate(cols[1:], start=2):
        ax.text(xs[j], y, cell(r, key), fontsize=11, color=INK, va="center", ha="right", transform=ax.transAxes)
    if i < 2:
        ax.plot([0.02, 0.995], [y - 0.10, y - 0.10], color=GRID, lw=0.8, transform=ax.transAxes)
ax.text(0.02, -0.06, "Headline basis (NHS England, including estimates for non-submitting trusts). "
        "Source: NHS England RTT, Aug 2026 release; rebuilt from provider files.",
        fontsize=7.8, color=INK2, transform=ax.transAxes)
fig.subplots_adjust(left=0.01, right=0.99, top=0.98, bottom=0.12)
fig.savefig(OUT / "three_row_table.png", dpi=200, facecolor=SURFACE)
plt.close(fig)
hb.to_csv(OUT / "three_row_table_headline.csv")
print("wrote", (OUT / "three_row_table.png").name)

wrote three_row_table.png


## Step 3 — Alt text, generated from the same numbers

Medium supports alt text on images. I write it from Gold so it can't drift
from the picture.

In [5]:
o = hb.loc["Over 52 weeks"]; tl = hb.loc["Total list"]; o18 = hb.loc["Over 18 weeks"]
alt_chart = (f"Line chart, April 2025 to August 2026, indexed to March 2026 = 100. The number waiting over 52 weeks falls from about "
             f"{chart.over_52.iloc[0]:.0f} in April 2025 to 100 at the March 2026 target deadline, then rises every month to "
             f"{chart.over_52.iloc[-1]:.1f} in August. The total list and the number waiting over 18 weeks finish at "
             f"{chart.total.iloc[-1]:.1f} and {chart.over_18.iloc[-1]:.1f}.")
alt_table = (f"Table. August 2026: total list {tl.aug_2026:,.0f} ({tl.since_mar_pct:+.1f}% since March, {tl.yoy_pct:+.1f}% on a year); "
             f"over 18 weeks {o18.aug_2026:,.0f} ({o18.since_mar_pct:+.1f}% since March, {o18.yoy_pct:+.1f}% on a year); "
             f"over 52 weeks {o.aug_2026:,.0f} ({o.since_mar_pct:+.1f}% since March, {o.mom_pct:+.1f}% on July, {o.yoy_pct:+.1f}% on a year).")
(OUT / "alt_text.md").write_text(f"# Alt text\n\n## divergence_since_march.png\n\n{alt_chart}\n\n## three_row_table.png\n\n{alt_table}\n")
print(alt_chart, "\n\n", alt_table)

Line chart, April 2025 to August 2026, indexed to March 2026 = 100. The number waiting over 52 weeks falls from about 201 in April 2025 to 100 at the March 2026 target deadline, then rises every month to 120.1 in August. The total list and the number waiting over 18 weeks finish at 103.7 and 104.0. 

 Table. August 2026: total list 7,370,100 (+3.7% since March, -0.5% on a year); over 18 weeks 2,564,009 (+4.0% since March, -11.1% on a year); over 52 weeks 113,339 (+20.1% since March, +1.8% on July, -40.5% on a year).
